# 1. Indexing

## 1.1 Load the document

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = Path.cwd().parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from arabic_legal_qa.rag.config import EmbeddingConfig, project_root
from arabic_legal_qa.rag.pdf_loader import EgyptianCivilCodeLoader
from arabic_legal_qa.rag.helper import stable_hash


ROOT = project_root(ROOT)
load_dotenv(ROOT / ".env", override=False)
PDF_PATH = ROOT / "data/raw/egyptian_civil_law.pdf"
PROCESSED_DIR = ROOT / "data/processed"

loader = EgyptianCivilCodeLoader(
    pdf_path=PDF_PATH,
    output_dir=PROCESSED_DIR,
    language="bilingual",
)

# One LangChain Document per validated Civil Code article.
documents = loader.load()

sample = next(
    doc for doc in documents
    if doc.metadata["article_number"] == 147 and doc.metadata["language"] == "ar"
)

print({
    "documents": len(documents),
    "articles": len(loader.articles or []),
    "valid": bool(loader.report and loader.report["valid"]),
    "sample_metadata": sample.metadata,
})

print(sample.page_content[:800])

## 1.2 Split the document into chunks

In [ ]:
from arabic_legal_qa.rag.llms import load_embedding_bundle
from arabic_legal_qa.rag.vec_db import ChunkingConfig, chunk_documents

embedding_config = EmbeddingConfig(
    model_name="intfloat/multilingual-e5-small",
    model_revision=os.getenv("E5_SMALL_REVISION") or None,
)
embedding_bundle = load_embedding_bundle(
    config=embedding_config,
    cache_dir=ROOT / "data/model_cache",
)

corpus_hash = stable_hash([doc.page_content for doc in documents])
# Leave room for E5's passage prefix and special tokens.
chunk_config = ChunkingConfig(embedding_limit=embedding_config.embedding_limit, chunk_tokens=480)
chunks = chunk_documents(
    documents=documents,
    tokenizer=embedding_bundle.tokenizer,
    config=chunk_config,
    corpus_hash=corpus_hash,
)

print({"chunks": len(chunks), "embedding_model": embedding_config.model_name})

## 1.3 Embed the chunks and store them in a vector database

In [ ]:
# The shared embedder is ready for the vector-store step.
# Qdrant indexing will be added here once the collection configuration is finalized.
embedder = embedding_bundle.embedder
print({"ready_for_indexing": True, "chunks": len(chunks)})

# 2. Retrieval-Augmented Generation (RAG)